# Multi-Agent E-commerce Dispute — standalone notebook

Notebook này chứa toàn bộ pipeline trong một file: nạp 50 case và 4 bảng Olist, chạy 6 agent, kiểm định output, ghi trace/metadata và tạo submission.zip.

Không có pip install, không gọi Hugging Face Hub và không tự tải model. Mặc định dùng rule engine xác định; Qwen chỉ được thử khi bạn chủ động bật ENABLE_LLM và chỉ rõ MODEL_PATH tới thư mục model đã attach/local.

In [ ]:
from collections import Counter, defaultdict
from datetime import datetime, timezone
from decimal import Decimal, ROUND_HALF_UP
import gc
import hashlib
import importlib.metadata as package_metadata
import json
import os
from pathlib import Path
import platform
import re
import time
import uuid
import zipfile

import pandas as pd

# User overrides. Leave None for safe discovery.
DATA_DIR = os.getenv('EC_DATA_DIR') or None
INPUT_DIR = os.getenv('EC_INPUT_DIR') or None
WORK_ROOT = os.getenv('EC_WORK_ROOT') or None
MODEL_PATH = os.getenv('QWEN_MODEL_PATH') or None
ENABLE_LLM = os.getenv('EC_ENABLE_LLM', '0').strip().lower() in {'1', 'true', 'yes', 'on'}

# Offline is enforced before any optional model import.
for _name in ('HF_HUB_OFFLINE', 'TRANSFORMERS_OFFLINE', 'HF_DATASETS_OFFLINE', 'HF_HUB_DISABLE_TELEMETRY'):
    os.environ[_name] = '1'

MODEL_ID = 'Qwen/Qwen3-8B'
MODEL_PARAMETER_SIZE = '8.2B'
POLICY_VERSION = 'EC_POLICY_V1'
EXPECTED_CASES = 50
OFFICIAL_CONFIDENCE = 0.92
MONEY_QUANTUM = Decimal('0.01')
RECONCILIATION_TOLERANCE = Decimal('0.10')
GOLDEN_PAYLOAD_SHA256 = 'd6f9007649523a48f11bde5e98a0166ebdbb87d6aadb92275f7e205d550e2e78'

ISSUE_SPECS = {
    'canceled_order_paid': ('ORDER_CANCELED_AFTER_PAYMENT', 'issue_full_refund'),
    'unavailable_order_paid': ('ORDER_UNAVAILABLE_AFTER_PAYMENT', 'issue_full_refund'),
    'late_delivery_seller': ('SELLER_HANDOFF_AFTER_LIMIT', 'refund_freight'),
    'late_delivery_logistics': ('CARRIER_DELIVERED_AFTER_ESTIMATE', 'refund_freight'),
    'valid_split_payment': ('MULTIPLE_PAYMENTS_RECONCILED', 'explain_valid_split_payment'),
    'unsupported_late_claim': ('DELIVERY_WITHIN_ESTIMATE', 'reject_late_refund'),
}
GOLDEN_ISSUES = {
    'canceled_order_paid': 8, 'unavailable_order_paid': 8,
    'late_delivery_seller': 8, 'late_delivery_logistics': 8,
    'valid_split_payment': 9, 'unsupported_late_claim': 9,
}
GOLDEN_STATUSES = {'action_required': 32, 'no_action': 18}
GOLDEN_TOTALS = {
    'item_total_brl': Decimal('4686.52'),
    'freight_total_brl': Decimal('727.47'),
    'payment_total_brl': Decimal('7782.89'),
    'recommended_refund_brl': Decimal('3429.64'),
}

DATA_FILES = (
    'olist_orders_dataset.csv', 'olist_order_items_dataset.csv',
    'olist_order_payments_dataset.csv', 'olist_sellers_dataset.csv',
)

def _valid_data_dir(path):
    return path.is_dir() and all((path / name).is_file() for name in DATA_FILES)

def _valid_input_dir(path):
    marker = path / 'EC_001.json'
    if not marker.is_file():
        return False
    try:
        payload = json.loads(marker.read_text(encoding='utf-8'))
        return payload.get('case_id') == 'EC_001' and 'customer_request' in payload
    except (OSError, json.JSONDecodeError, AttributeError):
        return False

def discover_dir(explicit, local_name, marker, validator):
    if explicit is not None:
        candidate = Path(explicit).expanduser().resolve()
        if validator(candidate):
            return candidate
        raise FileNotFoundError(f'Invalid explicit directory: {candidate}')
    for candidate in (Path.cwd() / local_name, Path.cwd(), Path.cwd().parent / local_name):
        if validator(candidate):
            return candidate.resolve()
    kaggle_input = Path('/kaggle/input')
    matches = [] if not kaggle_input.is_dir() else sorted({p.parent.resolve() for p in kaggle_input.rglob(marker) if validator(p.parent)}, key=str)
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        raise RuntimeError(f'Multiple directories contain {marker}; set the override explicitly: {matches}')
    raise FileNotFoundError(f'Cannot locate {marker}; set the corresponding override')

DATA_DIR = discover_dir(DATA_DIR, 'data', DATA_FILES[0], _valid_data_dir)
INPUT_DIR = discover_dir(INPUT_DIR, 'input', 'EC_001.json', _valid_input_dir)
WORK_ROOT = Path(WORK_ROOT or ('/kaggle/working' if Path('/kaggle').is_dir() else Path.cwd())).expanduser().resolve()
OUTPUT_DIR = WORK_ROOT / 'output'
LOGGING_DIR = WORK_ROOT / 'logging'
TRACE_PATH = WORK_ROOT / 'trace.jsonl'
METADATA_PATH = WORK_ROOT / 'metadata.json'
SUBMISSION_ZIP = WORK_ROOT / 'submission.zip'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOGGING_DIR.mkdir(parents=True, exist_ok=True)

print(json.dumps({
    'data_dir': str(DATA_DIR), 'input_dir': str(INPUT_DIR),
    'work_root': str(WORK_ROOT), 'model': MODEL_ID,
    'llm_requested': bool(ENABLE_LLM and MODEL_PATH),
    'model_downloads_allowed': False,
}, indent=2))


In [ ]:
def money(value):
    text = str(value).strip() if value is not None else '0'
    return Decimal(text or '0').quantize(MONEY_QUANTUM, rounding=ROUND_HALF_UP)

def sum_money(values):
    return money(sum((Decimal(str(value).strip() or '0') for value in values), Decimal('0')))

def money_float(value):
    return float(money(value))

def timestamp(value):
    if value is None or not str(value).strip():
        return None
    parsed = pd.to_datetime(str(value), format='%Y-%m-%d %H:%M:%S', errors='coerce')
    return None if pd.isna(parsed) else parsed

def json_safe(value):
    if isinstance(value, Decimal):
        return float(value)
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [json_safe(item) for item in value]
    if hasattr(value, 'item') and callable(value.item):
        return value.item()
    return value

def atomic_text(path, text):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_text(text, encoding='utf-8')
    os.replace(temporary, path)

def atomic_json(path, payload):
    atomic_text(path, json.dumps(json_safe(payload), ensure_ascii=False, indent=2, allow_nan=False) + '\n')

class TraceRecorder:
    def __init__(self):
        self.run_id = uuid.uuid4().hex
        self.started_at = datetime.now(timezone.utc)
        self.events = []

    def emit(self, case_id, agent, event, payload=None, from_agent=None, to_agent=None):
        record = {
            'run_id': self.run_id,
            'timestamp_utc': datetime.now(timezone.utc).isoformat(),
            'case_id': case_id, 'agent': agent, 'event': event,
        }
        if from_agent is not None:
            record['from_agent'] = from_agent
        if to_agent is not None:
            record['to_agent'] = to_agent
        if payload is not None:
            record['payload'] = json_safe(payload)
        self.events.append(record)

    def flush(self, *paths):
        content = ''.join(json.dumps(event, ensure_ascii=False, allow_nan=False) + '\n' for event in self.events)
        for path in dict.fromkeys(Path(item) for item in paths):
            atomic_text(path, content)

def read_csv(filename, columns):
    try:
        return pd.read_csv(DATA_DIR / filename, usecols=columns, dtype=str, keep_default_na=False)
    except (OSError, ValueError, pd.errors.ParserError) as exc:
        raise ValueError(f'Cannot read {filename}: {exc}') from exc

expected_names = {f'EC_{index:03d}.json' for index in range(1, EXPECTED_CASES + 1)}
case_paths = sorted(INPUT_DIR.glob('EC_*.json'))
actual_names = {path.name for path in case_paths}
if actual_names != expected_names:
    raise ValueError(f'Input set mismatch: missing={sorted(expected_names - actual_names)}, extra={sorted(actual_names - expected_names)}')

CASES = []
seen_case_ids, target_order_ids = set(), set()
for path in case_paths:
    case = json.loads(path.read_text(encoding='utf-8'))
    request = case.get('customer_request', {})
    case_id, order_id = case.get('case_id'), request.get('claimed_order_id')
    if case_id != path.stem or case.get('policy_version') != POLICY_VERSION:
        raise ValueError(f'Invalid case identity/policy in {path.name}')
    if not isinstance(order_id, str) or not order_id or case_id in seen_case_ids or order_id in target_order_ids:
        raise ValueError(f'Duplicate or invalid case/order in {path.name}')
    case['_source_filename'] = path.name
    CASES.append(case)
    seen_case_ids.add(case_id)
    target_order_ids.add(order_id)

orders = read_csv('olist_orders_dataset.csv', [
    'order_id', 'order_status', 'order_delivered_carrier_date',
    'order_delivered_customer_date', 'order_estimated_delivery_date',
])
items = read_csv('olist_order_items_dataset.csv', [
    'order_id', 'order_item_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value',
])
payments = read_csv('olist_order_payments_dataset.csv', [
    'order_id', 'payment_sequential', 'payment_type', 'payment_value',
])
sellers = read_csv('olist_sellers_dataset.csv', ['seller_id'])
source_counts = {'orders': len(orders), 'items': len(items), 'payments': len(payments), 'sellers': len(sellers)}

if orders['order_id'].duplicated().any():
    raise ValueError('orders.order_id must be unique')
if items[['order_id', 'order_item_id']].duplicated().any():
    raise ValueError('(order_id, order_item_id) must be unique')
if payments[['order_id', 'payment_sequential']].duplicated().any():
    raise ValueError('(order_id, payment_sequential) must be unique')
if sellers['seller_id'].duplicated().any():
    raise ValueError('sellers.seller_id must be unique')
if not items['order_item_id'].str.fullmatch(r'[1-9]\d*').all() or not payments['payment_sequential'].str.fullmatch(r'[1-9]\d*').all():
    raise ValueError('Item/payment sequence values must be positive integers')

known_order_ids = set(orders['order_id'])
if target_order_ids - known_order_ids:
    raise KeyError(f'Orders not found: {sorted(target_order_ids - known_order_ids)}')
known_seller_ids = set(sellers['seller_id'])
if set(items['seller_id']) - known_seller_ids:
    raise ValueError('order_items contains unknown seller_id values')

orders = orders.loc[orders['order_id'].isin(target_order_ids)].copy()
items = items.loc[items['order_id'].isin(target_order_ids)].copy()
payments = payments.loc[payments['order_id'].isin(target_order_ids)].copy()
timestamp_pattern = r'\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}'
for frame, columns, name in (
    (orders, ('order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date'), 'orders'),
    (items, ('shipping_limit_date',), 'items'),
):
    for column in columns:
        values = frame[column].str.strip()
        invalid = values.ne('') & ~values.str.fullmatch(timestamp_pattern)
        if invalid.any():
            raise ValueError(f'{name}.{column} lost Olist timestamp precision: {values[invalid].head().tolist()}')

def grouped(frame, sequence_column):
    result = defaultdict(list)
    for row in frame.to_dict(orient='records'):
        result[row['order_id']].append(row)
    return {order_id: tuple(sorted(rows, key=lambda row: int(row[sequence_column]))) for order_id, rows in result.items()}

ORDERS = {row['order_id']: row for row in orders.to_dict(orient='records')}
ITEMS_BY_ORDER = grouped(items, 'order_item_id')
PAYMENTS_BY_ORDER = grouped(payments, 'payment_sequential')
retained_counts = {
    'orders': len(orders), 'items': len(items), 'payments': len(payments),
    'sellers': len(set(items['seller_id'])),
}
print(f'Loaded {len(CASES)} cases; retained rows: {retained_counts}')


In [ ]:
class LocalQwenGateway:
    def __init__(self, enabled=False, model_path=None):
        self.requested = bool(enabled)
        self.path = Path(model_path).expanduser().resolve() if model_path else None
        self.status = 'disabled_deterministic_fallback' if not enabled else 'model_path_missing_fallback'
        self.ready = False
        self.calls = 0
        self.parsed = 0
        self.errors = []
        self.model = self.tokenizer = self.torch = None
        if enabled and self.path is not None:
            self._load()

    def _load(self):
        try:
            if not self.path.is_dir() or not (self.path / 'config.json').is_file():
                raise ValueError('MODEL_PATH is not a complete local model directory')
            config = json.loads((self.path / 'config.json').read_text(encoding='utf-8'))
            expected_config = {'model_type': 'qwen3', 'hidden_size': 4096, 'num_hidden_layers': 36, 'num_attention_heads': 32, 'num_key_value_heads': 8, 'eos_token_id': 151645}
            if not isinstance(config, dict) or any(config.get(key) != value for key, value in expected_config.items()):
                raise ValueError('MODEL_PATH is not the requested post-trained Qwen3-8B asset')
            import accelerate  # noqa: F401
            import bitsandbytes  # noqa: F401
            import torch
            from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
            if not torch.cuda.is_available():
                raise RuntimeError('CUDA unavailable; CPU loading is intentionally disabled')
            dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
            quantization = BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_quant_type='nf4',
                bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=dtype,
            )
            self.tokenizer = AutoTokenizer.from_pretrained(
                str(self.path), local_files_only=True, trust_remote_code=False,
            )
            self.model = AutoModelForCausalLM.from_pretrained(
                str(self.path), local_files_only=True, trust_remote_code=False,
                device_map={'': 0}, quantization_config=quantization, low_cpu_mem_usage=True,
            )
            if self.tokenizer.pad_token_id is None:
                self.tokenizer.pad_token_id = self.tokenizer.eos_token_id
            self.model.eval()
            self.torch, self.ready, self.status = torch, True, 'ready_local_nf4'
        except Exception as exc:
            self.errors.append(f'{type(exc).__name__}: {str(exc)[:300]}')
            self.status = 'model_load_fallback'
            self.close(relabel=False)

    @staticmethod
    def _parse_object(text):
        fence = chr(96) * 3
        cleaned = re.sub(r'<think>.*?</think>', '', text, flags=re.I | re.S).replace(fence + 'json', '').replace(fence, '')
        decoder = json.JSONDecoder()
        for index, character in enumerate(cleaned):
            if character == '{':
                try:
                    value, _ = decoder.raw_decode(cleaned[index:])
                    if isinstance(value, dict):
                        return value
                except json.JSONDecodeError:
                    pass
        return None

    def propose(self, facts):
        if not self.ready:
            return None, {'ok': False, 'reason': self.status}
        self.calls += 1
        system = (
            'Apply EC_POLICY_V1. Return exactly one JSON object with keys '
            'primary_issue, cause_code, party_type, party_id. Never invent values.'
        )
        messages = [{'role': 'system', 'content': system}, {'role': 'user', 'content': json.dumps(json_safe(facts))}]
        try:
            try:
                prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
            except TypeError:
                prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            tokens = self.tokenizer(prompt, return_tensors='pt')
            input_length = tokens['input_ids'].shape[-1]
            device = next(self.model.parameters()).device
            tokens = {key: value.to(device) for key, value in tokens.items()}
            with self.torch.inference_mode():
                generated = self.model.generate(
                    **tokens, max_new_tokens=64, do_sample=False, use_cache=True,
                    eos_token_id=self.tokenizer.eos_token_id,
                    pad_token_id=self.tokenizer.pad_token_id,
                )
            raw = self.tokenizer.decode(generated[0][input_length:], skip_special_tokens=True)
            candidate = self._parse_object(raw)
            self.parsed += candidate is not None
            return candidate, {'ok': candidate is not None, 'reason': None if candidate else 'parse_failed'}
        except Exception as exc:
            self.errors.append(f'{type(exc).__name__}: {str(exc)[:300]}')
            return None, {'ok': False, 'reason': 'generation_failed'}

    def snapshot(self):
        return {
            'requested': self.requested, 'status': self.status, 'ready': self.ready,
            'configured_model': MODEL_ID, 'source': str(self.path) if self.path else None,
            'network_permitted': False, 'download_policy': 'forbidden_local_files_only',
            'calls': self.calls, 'parsed_responses': self.parsed, 'errors': self.errors,
        }

    def close(self, relabel=True):
        self.ready, self.model, self.tokenizer = False, None, None
        gc.collect()
        if self.torch is not None:
            try:
                self.torch.cuda.empty_cache()
            except Exception:
                pass
        if relabel:
            self.status = 'closed'

class OrderSellerAgent:
    def __init__(self, trace):
        self.trace = trace

    def analyze(self, case):
        order_id = case['customer_request']['claimed_order_id']
        row = ORDERS[order_id]
        item_rows = ITEMS_BY_ORDER.get(order_id, ())
        item_facts = tuple({
            'item_id': f"{order_id}:{item['order_item_id']}",
            'seller_id': item['seller_id'], 'shipping_limit_date': item['shipping_limit_date'],
            'price_brl': money(item['price']), 'freight_brl': money(item['freight_value']),
        } for item in item_rows)
        sellers_in_order = tuple(dict.fromkeys(item['seller_id'] for item in item_rows))
        result = {
            'order_id': order_id, 'order_status': row['order_status'],
            'carrier_date': row['order_delivered_carrier_date'],
            'customer_date': row['order_delivered_customer_date'],
            'estimated_date': row['order_estimated_delivery_date'],
            'items': item_facts, 'seller_ids': sellers_in_order,
            'item_total_brl': sum_money(item['price'] for item in item_rows),
            'freight_total_brl': sum_money(item['freight_value'] for item in item_rows),
        }
        self.trace.emit(case['case_id'], 'OrderSellerAgent', 'handoff', {'order_id': order_id, 'item_count': len(item_facts)}, 'OrderSellerAgent', 'CoordinatorAgent')
        return result

class PaymentAgent:
    def __init__(self, trace):
        self.trace = trace

    def analyze(self, case, order):
        order_id = order['order_id']
        rows = PAYMENTS_BY_ORDER.get(order_id, ())
        facts = tuple({
            'payment_id': f"{order_id}:{row['payment_sequential']}",
            'payment_type': row['payment_type'], 'payment_value_brl': money(row['payment_value']),
        } for row in rows)
        total = sum_money(row['payment_value'] for row in rows)
        expected = money(order['item_total_brl'] + order['freight_total_brl'])
        result = {
            'order_id': order_id, 'payments': facts, 'payment_count': len(facts),
            'payment_total_brl': total, 'difference_brl': money(abs(total - expected)),
            'reconciled': abs(total - expected) <= RECONCILIATION_TOLERANCE,
        }
        self.trace.emit(case['case_id'], 'PaymentAgent', 'handoff', {'payment_count': len(facts), 'reconciled': result['reconciled']}, 'PaymentAgent', 'CoordinatorAgent')
        return result

class DeliveryAgent:
    def __init__(self, trace):
        self.trace = trace

    def analyze(self, case, order):
        delivered, estimated, carrier = timestamp(order['customer_date']), timestamp(order['estimated_date']), timestamp(order['carrier_date'])
        late = None if delivered is None or estimated is None else bool(delivered > estimated)
        limits = [(item['seller_id'], timestamp(item['shipping_limit_date'])) for item in order['items']]
        late_sellers = tuple(dict.fromkeys(seller for seller, limit in limits if carrier is not None and limit is not None and carrier > limit))
        complete = carrier is not None and bool(limits) and all(limit is not None for _, limit in limits)
        result = {
            'late_delivery': late, 'late_seller_ids': late_sellers,
            'carrier_handoff_on_time': complete and not late_sellers,
        }
        self.trace.emit(case['case_id'], 'DeliveryAgent', 'handoff', result, 'DeliveryAgent', 'CoordinatorAgent')
        return result

class PolicyAgent:
    def __init__(self, trace, gateway):
        self.trace, self.gateway = trace, gateway

    def decide(self, case, order, payment, delivery):
        status, paid = order['order_status'], payment['payment_total_brl']
        if status == 'canceled' and paid > 0:
            issue, refund = 'canceled_order_paid', paid
            parties = [{'party_type': 'platform', 'party_id': 'OLIST_PLATFORM'}]
        elif status == 'unavailable' and paid > 0:
            issue, refund = 'unavailable_order_paid', paid
            parties = [{'party_type': 'platform', 'party_id': 'OLIST_PLATFORM'}]
        elif delivery['late_delivery'] is True and delivery['late_seller_ids']:
            issue, refund = 'late_delivery_seller', order['freight_total_brl']
            parties = [{'party_type': 'seller', 'party_id': seller} for seller in delivery['late_seller_ids'][:3]]
        elif delivery['late_delivery'] is True and delivery['carrier_handoff_on_time']:
            issue, refund = 'late_delivery_logistics', order['freight_total_brl']
            parties = [{'party_type': 'logistics_provider', 'party_id': 'LOGISTICS_PROVIDER'}]
        elif payment['payment_count'] >= 2 and payment['reconciled']:
            issue, refund, parties = 'valid_split_payment', Decimal('0'), []
        elif delivery['late_delivery'] is False and payment['reconciled']:
            issue, refund, parties = 'unsupported_late_claim', Decimal('0'), []
        else:
            raise RuntimeError(f"{case['case_id']} is outside EC_POLICY_V1 coverage")
        cause, action = ISSUE_SPECS[issue]
        decision = {
            'issue': issue, 'cause': cause, 'action': action, 'parties': parties,
            'refund': money(refund), 'case_status': 'action_required' if money(refund) > 0 else 'no_action',
        }
        facts = {
            'order_status': status, 'payment_count': payment['payment_count'],
            'payment_total_brl': paid, 'reconciled': payment['reconciled'], **delivery,
        }
        candidate, model_meta = self.gateway.propose(facts)
        expected_candidate = {
            'primary_issue': issue, 'cause_code': cause,
            'party_type': parties[0]['party_type'] if len(parties) == 1 else None,
            'party_id': parties[0]['party_id'] if len(parties) == 1 else None,
        }
        accepted = candidate == expected_candidate
        decision['decision_source'] = 'qwen_validated' if accepted else 'deterministic_fallback'
        decision['fallback_reason'] = None if accepted else model_meta.get('reason', 'model_conflict')
        self.trace.emit(case['case_id'], 'PolicyAgent', 'decision', {'accepted': accepted, 'candidate': candidate, 'authoritative_issue': issue}, 'PolicyAgent', 'CoordinatorAgent')
        return decision

def assemble_output(case, order, payment, decision):
    item_ids = [item['item_id'] for item in order['items']][:5]
    seller_ids = list(order['seller_ids'][:5])
    payment_ids = [row['payment_id'] for row in payment['payments']][:5]
    evidence = [f"order:{order['order_id']}"]
    evidence += [f'item:{item_id}' for item_id in item_ids]
    evidence += [f'payment:{payment_id}' for payment_id in payment_ids]
    evidence += [f'seller:{seller_id}' for seller_id in seller_ids]
    evidence = evidence[:9] + [f"policy:{decision['cause']}"]
    return {
        'case_id': case['case_id'],
        'assessment': {
            'primary_issue': decision['issue'], 'case_status': decision['case_status'],
            'confidence': OFFICIAL_CONFIDENCE,
        },
        'affected_entities': {
            'order_ids': [order['order_id']], 'item_ids': item_ids,
            'seller_ids': seller_ids, 'payment_ids': payment_ids,
        },
        'root_cause_analysis': {
            'ranked_causes': [{'cause_code': decision['cause'], 'rank': 1}],
            'responsible_parties': decision['parties'],
        },
        'evidence_ids': evidence,
        'financial_resolution': {
            'currency': 'BRL', 'item_total_brl': money_float(order['item_total_brl']),
            'freight_total_brl': money_float(order['freight_total_brl']),
            'payment_total_brl': money_float(payment['payment_total_brl']),
            'recommended_refund_brl': money_float(decision['refund']),
        },
        'resolution_actions': [decision['action']],
    }

class VerifierAgent:
    ROOT_KEYS = {'case_id', 'assessment', 'affected_entities', 'root_cause_analysis', 'evidence_ids', 'financial_resolution', 'resolution_actions'}

    def __init__(self, trace):
        self.trace = trace

    def verify(self, case, payload, order, payment, decision):
        errors = []
        if set(payload) != self.ROOT_KEYS or payload['case_id'] != case['case_id']:
            errors.append('root_schema')
        if payload['assessment'] != {
            'primary_issue': decision['issue'], 'case_status': decision['case_status'],
            'confidence': OFFICIAL_CONFIDENCE,
        }:
            errors.append('assessment')
        expected_entities = {
            'order_ids': [order['order_id']],
            'item_ids': [item['item_id'] for item in order['items']][:5],
            'seller_ids': list(order['seller_ids'][:5]),
            'payment_ids': [row['payment_id'] for row in payment['payments']][:5],
        }
        if payload['affected_entities'] != expected_entities:
            errors.append('entities')
        if payload['root_cause_analysis'] != {
            'ranked_causes': [{'cause_code': decision['cause'], 'rank': 1}],
            'responsible_parties': decision['parties'],
        }:
            errors.append('root_cause')
        allowed = {f"order:{order['order_id']}", f"policy:{decision['cause']}"}
        allowed |= {f"item:{item['item_id']}" for item in order['items']}
        allowed |= {f"payment:{row['payment_id']}" for row in payment['payments']}
        allowed |= {f'seller:{seller_id}' for seller_id in order['seller_ids']}
        if len(payload['evidence_ids']) > 10 or len(set(payload['evidence_ids'])) != len(payload['evidence_ids']) or any(item not in allowed for item in payload['evidence_ids']):
            errors.append('evidence')
        expected_money = {
            'currency': 'BRL', 'item_total_brl': money_float(order['item_total_brl']),
            'freight_total_brl': money_float(order['freight_total_brl']),
            'payment_total_brl': money_float(payment['payment_total_brl']),
            'recommended_refund_brl': money_float(decision['refund']),
        }
        if payload['financial_resolution'] != expected_money or payload['resolution_actions'] != [decision['action']]:
            errors.append('resolution')
        try:
            json.dumps(payload, ensure_ascii=False, allow_nan=False)
        except (TypeError, ValueError):
            errors.append('strict_json')
        if errors:
            raise ValueError(f"Verifier rejected {case['case_id']}: {errors}")
        self.trace.emit(case['case_id'], 'VerifierAgent', 'verification', {'valid': True, 'errors': []}, 'VerifierAgent', 'CoordinatorAgent')
        return payload

class CoordinatorAgent:
    def __init__(self, trace, gateway):
        self.trace = trace
        self.order = OrderSellerAgent(trace)
        self.payment = PaymentAgent(trace)
        self.delivery = DeliveryAgent(trace)
        self.policy = PolicyAgent(trace, gateway)
        self.verifier = VerifierAgent(trace)

    def process(self, case):
        case_id = case['case_id']
        self.trace.emit(case_id, 'CoordinatorAgent', 'case_started', {'source_file': case['_source_filename']})
        self.trace.emit(case_id, 'CoordinatorAgent', 'dispatch', {'scope': ['claimed_order_id']}, 'CoordinatorAgent', 'OrderSellerAgent')
        order = self.order.analyze(case)
        self.trace.emit(case_id, 'CoordinatorAgent', 'dispatch', {'scope': ['item_total_brl', 'freight_total_brl']}, 'CoordinatorAgent', 'PaymentAgent')
        payment = self.payment.analyze(case, order)
        self.trace.emit(case_id, 'CoordinatorAgent', 'dispatch', {'scope': ['delivery_timestamps', 'shipping_limits']}, 'CoordinatorAgent', 'DeliveryAgent')
        delivery = self.delivery.analyze(case, order)
        decision = self.policy.decide(case, order, payment, delivery)
        output = self.verifier.verify(case, assemble_output(case, order, payment, decision), order, payment, decision)
        self.trace.emit(case_id, 'CoordinatorAgent', 'case_completed', {
            'primary_issue': decision['issue'], 'decision_source': decision['decision_source'],
            'recommended_refund_brl': decision['refund'],
        })
        return output, decision


In [ ]:
started = time.perf_counter()
trace = TraceRecorder()
gateway = LocalQwenGateway(ENABLE_LLM, MODEL_PATH)
trace.emit(None, 'QwenGateway', 'backend_initialized', gateway.snapshot())
coordinator = CoordinatorAgent(trace, gateway)

for stale in OUTPUT_DIR.glob('EC_*.json'):
    stale.unlink()

results, decisions = {}, {}
for index, case in enumerate(CASES, start=1):
    output, decision = coordinator.process(case)
    atomic_json(OUTPUT_DIR / case['_source_filename'], output)
    results[case['case_id']] = output
    decisions[case['case_id']] = decision
    if index % 10 == 0:
        print(f'Processed {index}/{len(CASES)} cases')

# Disk/canonical and golden hard gates.
output_paths = sorted(OUTPUT_DIR.glob('EC_*.json'))
if {path.name for path in output_paths} != expected_names:
    raise AssertionError('Output directory must contain exactly EC_001.json ... EC_050.json')
disk_results = {}
for path in output_paths:
    payload = json.loads(path.read_text(encoding='utf-8'))
    if payload.get('case_id') != path.stem or payload != results[path.stem]:
        raise AssertionError(f'Disk/canonical mismatch in {path.name}')
    disk_results[path.stem] = payload

issue_counts = Counter(payload['assessment']['primary_issue'] for payload in disk_results.values())
status_counts = Counter(payload['assessment']['case_status'] for payload in disk_results.values())
confidence_counts = Counter(str(payload['assessment']['confidence']) for payload in disk_results.values())
aggregate_totals = {
    key: money(sum((Decimal(str(payload['financial_resolution'][key])) for payload in disk_results.values()), Decimal('0')))
    for key in GOLDEN_TOTALS
}
if issue_counts != Counter(GOLDEN_ISSUES):
    raise AssertionError(f'Issue profile mismatch: {issue_counts}')
if status_counts != Counter(GOLDEN_STATUSES):
    raise AssertionError(f'Status profile mismatch: {status_counts}')
if confidence_counts != Counter({'0.92': EXPECTED_CASES}):
    raise AssertionError(f'Confidence profile mismatch: {confidence_counts}')
if aggregate_totals != GOLDEN_TOTALS:
    raise AssertionError(f'Money totals mismatch: {aggregate_totals}')
payload_sha256 = hashlib.sha256(b''.join(path.read_bytes() for path in output_paths)).hexdigest()
if payload_sha256 != GOLDEN_PAYLOAD_SHA256:
    raise AssertionError(f'Per-case payload regression: {payload_sha256}')

temporary_zip = SUBMISSION_ZIP.with_name(SUBMISSION_ZIP.name + '.tmp')
with zipfile.ZipFile(temporary_zip, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in output_paths:
        archive.write(path, arcname=path.name)
with zipfile.ZipFile(temporary_zip) as archive:
    if archive.namelist() != [f'EC_{index:03d}.json' for index in range(1, EXPECTED_CASES + 1)]:
        raise AssertionError('ZIP names/order mismatch')
    if any('/' in name or '\\' in name for name in archive.namelist()):
        raise AssertionError('ZIP entries must be at root')
    for name in archive.namelist():
        if json.loads(archive.read(name)) != disk_results[Path(name).stem]:
            raise AssertionError(f'ZIP/disk mismatch in {name}')
os.replace(temporary_zip, SUBMISSION_ZIP)

duration = round(time.perf_counter() - started, 3)
qwen_count = sum(decision['decision_source'] == 'qwen_validated' for decision in decisions.values())
trace.emit(None, 'CoordinatorAgent', 'run_completed', {
    'cases_processed': len(results), 'duration_seconds': duration,
    'qwen_validated_cases': qwen_count, 'qa_passed': True,
})
if len(trace.events) != 502:
    raise AssertionError(f'Expected 502 trace events, got {len(trace.events)}')
trace.flush(TRACE_PATH, LOGGING_DIR / 'trace.jsonl')

def installed_version(name):
    try:
        return package_metadata.version(name)
    except package_metadata.PackageNotFoundError:
        return None

model_snapshot = gateway.snapshot()
qa_report = {
    'issue_counts': dict(issue_counts), 'status_counts': dict(status_counts),
    'confidence_counts': dict(confidence_counts),
    'aggregate_totals': {key: str(value) for key, value in aggregate_totals.items()},
    'output_count': len(disk_results), 'payload_sha256': payload_sha256,
}
metadata = {
    'project': {'name': 'multiagent-a2a-standalone', 'version': '2.0.0'},
    'model': MODEL_ID, 'parameter_size': MODEL_PARAMETER_SIZE,
    'model_limit_compliance': '8.2B <= 10B per agent',
    'policy_version': POLICY_VERSION, 'llm_backend': model_snapshot,
    'framework': {
        'orchestration': 'standalone Python structured-handoff multi-agent',
        'inference': 'Transformers optional/local-only',
        'pandas_version': pd.__version__, 'transformers_version': installed_version('transformers'),
    },
    'runtime': {
        'environment': 'kaggle' if Path('/kaggle').is_dir() else 'local',
        'python_version': platform.python_version(), 'platform': platform.platform(),
        'started_at_utc': trace.started_at.isoformat(), 'duration_seconds': duration,
    },
    'data': {'source_row_counts': source_counts, 'retained_row_counts': retained_counts},
    'run': {
        'run_id': trace.run_id, 'cases_processed': len(results),
        'qwen_validated_cases': qwen_count,
        'deterministic_fallback_cases': len(results) - qwen_count,
        'trace_events': len(trace.events), 'qa': qa_report,
    },
}
atomic_json(METADATA_PATH, metadata)
atomic_json(LOGGING_DIR / 'metadata.json', metadata)
gateway.close()

print(json.dumps({
    'cases_processed': len(results), 'duration_seconds': duration,
    'confidence_counts': dict(confidence_counts),
    'payload_sha256': payload_sha256, 'submission_zip': str(SUBMISSION_ZIP),
}, ensure_ascii=False, indent=2))


In [ ]:
metadata_check = json.loads(METADATA_PATH.read_text(encoding='utf-8'))
with zipfile.ZipFile(SUBMISSION_ZIP) as archive:
    zip_names = archive.namelist()
audit = {
    'zip_path': str(SUBMISSION_ZIP),
    'zip_sha256': hashlib.sha256(SUBMISSION_ZIP.read_bytes()).hexdigest(),
    'zip_entries': len(zip_names),
    'zip_root_only': all('/' not in name and '\\' not in name for name in zip_names),
    'payload_sha256': metadata_check['run']['qa']['payload_sha256'],
    'confidence_counts': metadata_check['run']['qa']['confidence_counts'],
    'trace_events': metadata_check['run']['trace_events'],
    'model_status': metadata_check['llm_backend']['status'],
    'model_network_permitted': metadata_check['llm_backend']['network_permitted'],
}
print(json.dumps(audit, ensure_ascii=False, indent=2))

try:
    from IPython.display import FileLink, display
    display(FileLink(str(SUBMISSION_ZIP)))
except ImportError:
    print(SUBMISSION_ZIP)


## Trước khi submit

- Run All không lỗi.
- Audit phải có zip_entries = 50, zip_root_only = true, confidence_counts = {0.92: 50}, trace_events = 502 và model_network_permitted = false.
- Chỉ submit submission.zip. Không đưa trace, metadata hoặc notebook vào ZIP.